# Deep Research with OpenShell

A lead agent delegates competitor research, checks the findings and prepares a sourced comparison. OpenShell controls the inner agent's filesystem and network access. This notebook is the trusted control surface, outside that policy.

Your API keys come from the Devlab creation form. Do not paste them into notebook cells, print the environment or share your Jupyter token. Research does not start at Devlab startup. Running the research cell below uses your Token Factory and Tavily accounts and may incur charges.

Run the cells in order. The research source and downloaded artifacts live in the persistent `/workspace/research` directory. See [third-party notices](THIRD_PARTY_NOTICES.md) for orjson and certifi license and source information.

In [ ]:
from pathlib import Path
import subprocess
import sys
import uuid

project = Path('/workspace/research')

def openshell(*arguments):
    return subprocess.run(
        [sys.executable, 'openshell/run.py', *arguments],
        cwd=project,
        check=True,
    )


## 1. Check the runtime and register credentials

The Devlab starts its own Docker daemon and a local OpenShell gateway. This command checks the runtime and creates or refreshes the two credential providers. The inner agent receives credential placeholders rather than real keys. This step does not run research. Research dependencies are installed only inside the sandbox image.

In [ ]:
openshell('setup')


## 2. Run a research question

Edit the question before running this cell. Each run gets a new sandbox. The wrapper builds the locked agent image, runs the agent through OpenShell, then downloads its files into `runs/<run-id>/`. The sandbox stays available for inspection.

The inner sandbox defaults to 2 CPU and 4 GiB of memory; the outer Devlab also needs resources for Jupyter, Docker and the gateway. The first image build may take several minutes.

In [ ]:
question = 'Compare Linear against Jira and Asana'
sandbox_name = 'dr-' + uuid.uuid4().hex[:12]
print(f'Sandbox: {sandbox_name}')
openshell('run', '--name', sandbox_name, question)


## 3. Read the results and inspect policy decisions

Open the `runs` directory in the JupyterLab file browser. The wrapper prints the exact run directory. Read `final.md`, then inspect the generated notes, findings and `tool_calls.jsonl`; review source links before relying on the conclusions.

The policy permits Python to send only `POST /v1/chat/completions` to Nebius and `POST /search` to Tavily. Direct access to result URLs is not allowed. Tavily is separately instructed to exclude X/Twitter and Reddit from its returned results.

In [ ]:
openshell('inspect', sandbox_name)


## 4. Try denied requests

The demo attempts an unlisted host, an allowed host from an unapproved executable and an unapproved HTTP method. Check the matching policy-denial audit events in the printed logs. A failed command alone is not proof: DNS, TLS or application failures can also return a nonzero exit status.

In [ ]:
openshell('demo', sandbox_name)


## 5. Delete the sandbox when finished

Retained sandboxes keep using resources. After checking that the files were downloaded, set `delete_sandbox` to `True` and run the cell. Downloaded files in `runs/` remain available. Deleting the Devlab's workspace removes those files and its saved OpenShell state.

In [ ]:
delete_sandbox = False
if delete_sandbox:
    openshell('delete', sandbox_name)
